# INF201 – Deliverable 2: Campus Coffee Shop Sales Pipeline

**Group members:** Mathias Gundersen, Jakub Mermer  
**GitHub repository:** https://github.com/saltonese/Del_2-G6_5.git

## Introduction and approach

This notebook implements an ETL pipeline for the cash-register logs of the Campus Coffee chain:

1. **Extract** – recursively find all `.log` files, read them with the correct encoding (UTF-8 or ISO-8859-1) and pick out the valid sales lines with a regular expression.
2. **Transform** – build a DataFrame, remove price glitches with NumPy, and correct NOK prices that were registered on the EUR scale (× 10).
3. **Load / report** – hourly revenue per item, a status pivot table and refund ratios, written to `output/`.

Tasks 1–4 are small, individually tested building blocks. Task 5 connects them together into one .

### Interpretation choices

The assignment leaves a few things open. These are the choices made here:


### How to run
1. `python generate_data_for_d2.py` (creates `data/`; `BASE_DIR` was changed to `Path("Data")` and `random.seed(42)` added so all group members get identical data).
2. Run this notebook from the same folder. Output files are written to `output/`.

In [19]:
import json
import re
import tempfile #used to make temporary files, usefull when testing along the way
from pathlib import Path #used to easyely path between files and directories

import matplotlib.pyplot as plt #used to plot graphs
import numpy as np 
import pandas as pd

print(Path.cwd())
DATA_DIR = Path.cwd().parent / "Data"
OUTPUT_DIR = Path("output")

EPSILON = 1e-6          # keeps the refund ratio finite when sales_count == 0
NOK_CORRECTION = 10.0   # NOK prices were registered on the EUR scale
OUTLIER_N_STD = 3.0     # |x - mean| > 3 * std  ->  hardware glitch

print(f"pandas {pd.__version__} | numpy {np.__version__}")
assert DATA_DIR.exists(), "Run `python generate_data_for_d2.py` first to create the Data/ folder."

c:\Users\mathi\StudieNMBU\PROG\Del_2-G6_5\Notebooks
pandas 2.3.3 | numpy 2.3.4


## Task 1 – File ingestion and regular expressions

### 1.1 Robust file reader
The file is read as raw **bytes** and decoded explicitly. Reading bytes (instead of `read_text`) also avoids platform-dependent newline translation.

* First try `utf-8` (strict). Valid UTF-8 text is easy to recognise: Latin-1 text containing `é` or `ø` is almost never valid UTF-8, so a wrong guess raises `UnicodeDecodeError`.
* If that fails, decode as `iso-8859-1`. This always succeeds because every byte value maps to a character – that is why it has to be the *second* attempt.

Note: pure-ASCII files decode identically in both encodings, so they are reported as `utf-8`. The test below uses `é` and `ø` to show that the fallback really works.

In [20]:
def read_log_file(path: str | Path) -> tuple[str, str]:
    """Read a text file whose encoding is either UTF-8 or ISO-8859-1.

    Returns
    -------
    (file_text_content, encoding_used)
    """
    path = Path(path)
    raw = path.read_bytes()
    try:
        return raw.decode("utf-8"), "utf-8"
    except UnicodeDecodeError:
        return raw.decode("iso-8859-1"), "iso-8859-1"

In [21]:
# Test: the same text saved with two different encodings
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    (tmp / "utf8.log").write_bytes("Café ølbar\n".encode("utf-8"))
    (tmp / "latin1.log").write_bytes("Café ølbar\n".encode("iso-8859-1"))

    text_utf8, enc_utf8 = read_log_file(tmp / "utf8.log")
    text_latin1, enc_latin1 = read_log_file(tmp / "latin1.log")

assert (enc_utf8, enc_latin1) == ("utf-8", "iso-8859-1")
assert text_utf8 == text_latin1 == "Café ølbar\n"
print("read_log_file OK:", enc_utf8, "/", enc_latin1)

read_log_file OK: utf-8 / iso-8859-1


### 1.2 Log parser with regular expressions
One compiled pattern with **named groups** matches a complete valid line. Details:

* `^ … $` anchor the pattern to the whole line, so a noise line such as `[DEBUG] BARCODE SCANNER REBOOTED` (which also starts with `[`) cannot match by accident.
* `item=(?P<item>.+?)\s+price=` is lazy, so the item name can in principle contain spaces ("Iced Latte").
* `status` is restricted to the three allowed values, so a corrupted status makes the whole line invalid instead of slipping through.
* Noise lines simply do not match and are skipped; the price is converted to `float`.

In [22]:
LOG_PATTERN = re.compile(
    r"""
    ^\[(?P<timestamp>\d{4}-\d{2}-\d{2}\ \d{2}:\d{2}:\d{2})\]\s+   # [2026-10-15 08:30:00]
    \[(?P<register_id>[A-Za-z0-9_]+)\]\s+                       # [POS_01]
    item=(?P<item>.+?)\s+                                       # item=Latte
    price=(?P<price>\d+(?:\.\d+)?)\s+                           # price=4.50
    (?P<currency>[A-Z]{3})\s+                                   # EUR / NOK / USD
    \((?P<status>SUCCESS|REFUND|CANCELLED)\)$                   # (SUCCESS)
    """,
    re.VERBOSE,
)

# Only used for the validation at the end (the parser itself ignores everything that does not match)
NOISE_PATTERN = re.compile(r"^(---.*---|===.*===|\[DEBUG\].*)$")


def parse_log_text(text: str) -> list[dict]:
    """Extract all valid sales lines from log text, ignoring system noise.

    Each record has the keys: timestamp (str), register_id (str), item (str),
    price (float), currency (str), status (str).
    """
    records = []
    for line in text.splitlines():
        match = LOG_PATTERN.match(line.strip())
        if match is None:
            continue  # noise or malformed line
        record = match.groupdict()
        record["price"] = float(record["price"])
        records.append(record)
    return records

In [23]:
sample = """--- PRINTER PAPER LOW ---
[2026-10-15 08:30:00] [POS_01] item=Latte price=4.50 EUR (SUCCESS)
[DEBUG] BARCODE SCANNER REBOOTED
=== NETWORK SYNC OK ===
[2026-10-15 08:31:12] [POS_02] item=Muffin price=999.00 NOK (REFUND)
[2026-10-15 08:32:00] [POS_02] item=Muffin price=abc NOK (REFUND)
[2026-10-15 08:33:00] [POS_02] item=Muffin price=3.50 NOK (UNKNOWN)
"""

parsed = parse_log_text(sample)
assert len(parsed) == 2
assert parsed[0] == {
    "timestamp": "2026-10-15 08:30:00",
    "register_id": "POS_01",
    "item": "Latte",
    "price": 4.5,
    "currency": "EUR",
    "status": "SUCCESS",
}
assert parsed[1]["price"] == 999.0 and parsed[1]["status"] == "REFUND"
print("parse_log_text OK ->", len(parsed), "of 7 lines are valid")

parse_log_text OK -> 2 of 7 lines are valid


## Task 2 – Recursive utility functions

### 2.1 Recursive search for log files
Explicit recursion: for every entry in a directory, a sub-directory triggers a recursive call and a file with the right extension is collected. No `glob`, `rglob` or `os.walk`. The base case is implicit – a directory without sub-directories simply makes no further recursive calls.

`iterdir()` returns entries in arbitrary order, so the workflow sorts the result to make the pipeline deterministic.

In [24]:
def find_log_files(directory: str | Path, extension: str = ".log") -> list[Path]:
    """Recursively collect all files with the given extension below `directory`."""
    directory = Path(directory)
    found: list[Path] = []
    for item in directory.iterdir():
        if item.is_dir():
            found.extend(find_log_files(item, extension))   # recursive case
        elif item.is_file() and item.suffix == extension:
            found.append(item)
    return found

In [25]:
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    for rel in ["2026/a/POS_01/x.log", "2026/a/POS_02/y.log", "2026/b/POS_01/z.log",
                "2026/b/POS_01/notes.txt", "top.log"]:
        file = tmp / rel
        file.parent.mkdir(parents=True, exist_ok=True)
        file.write_text("test")
    found = find_log_files(tmp)

assert sorted(p.name for p in found) == ["top.log", "x.log", "y.log", "z.log"]
assert all(isinstance(p, Path) for p in found)
print("find_log_files OK ->", sorted(p.name for p in found))

find_log_files OK -> ['top.log', 'x.log', 'y.log', 'z.log']


### 2.2 Recursive traversal of the nested config
The JSON can be any mix of dicts and lists, so the function branches on the *type* of the current node:

* **dict** – if it has `"active": true` and a string `"id"`, record the id; then recurse into every value.
* **list** – recurse into every element.
* **anything else** (str, int, bool, None) – base case, nothing to collect.

The check `is True` makes sure only a real boolean `true` counts (not e.g. the string `"yes"`).

In [26]:
def get_active_registers(node) -> set[str]:
    """Return the ids of all registers with "active": true anywhere in a nested dict/list structure."""
    active: set[str] = set()
    if isinstance(node, dict):
        if node.get("active") is True and isinstance(node.get("id"), str):
            active.add(node["id"])
        for value in node.values():
            active |= get_active_registers(value)
    elif isinstance(node, list):
        for element in node:
            active |= get_active_registers(element)
    return active    # scalars: base case -> empty set

In [28]:
example_config = {
    "company": "Campus Coffee",
    "regions": [{
        "region_name": "Main Campus",
        "stores": [{
            "name": "Library Cafe",
            "registers": [
                {"id": "POS_01", "active": True},
                {"id": "POS_02", "active": False}],
        }],
    }],
}
assert get_active_registers(example_config) == {"POS_01"}

with open(DATA_DIR / "store_config.json", encoding="utf-8") as f:
    store_config = json.load(f)
print("get_active_registers OK -> active in store_config.json:", sorted(get_active_registers(store_config)))

get_active_registers OK -> active in store_config.json: ['POS_01', 'POS_02', 'POS_03']
